# 01 — Origins and Mathematical Foundations of Physics-Informed Neural Networks

> *"The laws of nature are but the mathematical thoughts of God."* — Euclid

This notebook reconstructs the genealogy of **Physics-Informed Neural Networks (PINNs)** from absolute first principles. We begin not with neural networks, but with the central question of computational physics: *how does one approximate the solution of a partial differential equation (PDE) on a computer?*  We then trace the parallel evolution of statistical learning, and finally show how the two threads converge in the PINN framework introduced by Raissi, Perdikaris, and Karniadakis (2019).

The reader is assumed comfortable with functional analysis ($L^p$, Sobolev spaces), operator theory on Hilbert spaces, the variational calculus, and graduate-level mathematical physics.

---

## Table of Contents

1. The Central Problem of Computational Physics
2. Classical Discretization I — Finite Difference Methods
3. Classical Discretization II — Finite Element Methods and Weak Forms
4. The Curse of the Mesh
5. Neural Networks as Continuous Function Approximators
6. The Universal Approximation Theorem — Statement and Proof Sketch
7. Function Spaces, Hilbert Spaces, and the Geometry of Solutions
8. From Strong Form to Optimization — The Calculus of Variations
9. The PINN Ansatz — Embedding Differential Operators in $C^\infty$
10. A First Look at the Composite Functional
11. Convergence, Consistency, and the Limits of the Paradigm

## 1. The Central Problem of Computational Physics

Let $\Omega \subset \mathbb{R}^d$ be an open, bounded domain with sufficiently regular boundary $\partial\Omega$, and let $T > 0$. A vast class of physical phenomena is governed by a PDE of the abstract form

$$
\mathcal{N}[u](x,t) = f(x,t), \qquad (x,t) \in \Omega \times (0,T],
$$

subject to initial conditions $u(x,0) = u_0(x)$ on $\Omega$ and boundary conditions $\mathcal{B}[u](x,t) = g(x,t)$ on $\partial\Omega \times [0,T]$, where $\mathcal{N}$ is a (possibly nonlinear) differential operator and $\mathcal{B}$ a boundary operator (Dirichlet, Neumann, Robin, or mixed).

Concrete instances span the entirety of mathematical physics:

| Equation | Operator $\mathcal{N}$ | Physics |
|---|---|---|
| Heat / diffusion | $\partial_t - \alpha \Delta$ | Fourier conduction |
| Wave / d'Alembert | $\partial_t^2 - c^2 \Delta$ | Acoustic and EM propagation |
| Schrödinger | $i\hbar\,\partial_t + \tfrac{\hbar^2}{2m}\Delta - V$ | Non-relativistic quantum |
| Navier–Stokes | $\partial_t + (u\cdot\nabla) - \nu\Delta + \tfrac{1}{\rho}\nabla p$ | Incompressible flow |
| Burgers | $\partial_t + u\,\partial_x - \nu \partial_x^2$ | 1-D shock & turbulence toy |
| Einstein field eqns. | $R_{\mu\nu} - \tfrac{1}{2}R g_{\mu\nu}$ | General relativity |

The *forward problem* asks: given $\mathcal{N}, f, u_0, g$, find $u$. The *inverse problem* asks: given measurements $\{u(x_i,t_i)\}$, infer parameters inside $\mathcal{N}$ (e.g. the viscosity $\nu$, the diffusivity $\alpha$, the potential $V$).

Except in pathologically symmetric regimes, **closed-form solutions do not exist**. The discipline of *numerical analysis of PDEs* exists precisely to fill this gap, and its dominant strategies — finite differences and finite elements — are best understood as two distinct philosophies of discretization.

## 2. Classical Discretization I — Finite Difference Methods (FDM)

Finite differences are the oldest tool in the kit, traceable to Euler's 1768 *Institutionum calculi integralis*. The idea is brutally direct: replace continuous derivatives by *difference quotients* on a regular lattice.

Given a uniform grid $x_i = i\,h$, $i = 0,\dots,N$, with spacing $h = L/N$, and recalling Taylor's theorem

$$
u(x_i \pm h) = u(x_i) \pm h\,u'(x_i) + \tfrac{h^2}{2}u''(x_i) \pm \tfrac{h^3}{6}u'''(x_i) + \tfrac{h^4}{24}u^{(4)}(x_i) + \mathcal{O}(h^5),
$$

one obtains the canonical stencils:

$$
u'(x_i) \;\approx\; \frac{u_{i+1} - u_{i-1}}{2h} + \mathcal{O}(h^2), \qquad
u''(x_i) \;\approx\; \frac{u_{i+1} - 2u_i + u_{i-1}}{h^2} + \mathcal{O}(h^2).
$$

The PDE is enforced *pointwise* on the lattice, producing a (sparse, structured) algebraic system $A\,\mathbf{u} = \mathbf{b}$.

### 2.1 Consistency, Stability, and Convergence

The Lax–Richtmyer theorem distills FDM's mathematical content into a triple:

$$
\boxed{\text{Consistency} \;+\; \text{Stability} \;\Longleftrightarrow\; \text{Convergence}.}
$$

Concretely, for an explicit scheme $u^{n+1} = G(u^n)$ approximating $\partial_t u = \mathcal{L}u$:

* *Consistency*: $\|G(u(\cdot,t_n)) - u(\cdot,t_{n+1})\| = o(\Delta t)$ as $h,\Delta t \to 0$.
* *Stability* (von Neumann): for the linearized scheme with Fourier symbol $\hat G(\xi)$, $|\hat G(\xi)| \le 1 + C\Delta t$ uniformly in $\xi$.
* The CFL condition $\Delta t \le C\,h^p/\|\mathcal{L}\|$ is the practical manifestation.

### 2.2 Strengths and Inherent Limitations

* **Strengths.** Trivially parallel, locally explicit, easy to analyze spectrally, well-suited to rectangular domains.
* **Limitations.** 
  * Complex geometry forces *body-fitted* or *immersed-boundary* hacks.
  * High-order stencils widen, costing locality.
  * Curse of dimensionality: $N^d$ unknowns in $d$ dimensions.
  * Solution is known only on lattice points — interpolation is post-hoc, not native.
  * Boundary regularity below $C^k$ destroys formal order of accuracy.

## 3. Classical Discretization II — Finite Element Methods (FEM)

FEM, formalized by Courant (1943) and operationalized by Argyris, Clough, Zienkiewicz in the 1950s–60s, abandons pointwise enforcement in favor of an integral (variational, or *weak*) formulation. The mathematical heart of FEM lies in functional analysis on Sobolev spaces.

### 3.1 The Weak Formulation

Consider the prototypical Poisson problem $-\Delta u = f$ on $\Omega$, $u|_{\partial\Omega} = 0$. Multiply by a *test function* $v \in H^1_0(\Omega)$ and integrate by parts:

$$
\int_\Omega \nabla u \cdot \nabla v\,d\Omega \;=\; \int_\Omega f\,v\,d\Omega \qquad \forall v \in H^1_0(\Omega).
$$

Define

$$
a(u,v) := \int_\Omega \nabla u\cdot\nabla v\,d\Omega, \qquad \ell(v) := \int_\Omega f\,v\,d\Omega.
$$

The weak problem reads: *find $u \in H^1_0(\Omega)$ such that $a(u,v) = \ell(v)$ for every $v \in H^1_0(\Omega)$.* By the **Lax–Milgram theorem**, if $a$ is bilinear, continuous, and coercive on a Hilbert space $V$, and $\ell \in V^\ast$, then the weak problem has a unique solution depending continuously on $f$.

### 3.2 Galerkin Projection

Choose a finite-dimensional subspace $V_h \subset V$ spanned by basis functions $\{\phi_j\}_{j=1}^N$ (typically piecewise polynomials on a triangulation $\mathcal{T}_h$). Seek $u_h = \sum_j c_j \phi_j$ with

$$
a(u_h, \phi_i) = \ell(\phi_i) \qquad i = 1,\dots,N,
$$

yielding the linear system $K\,\mathbf{c} = \mathbf{F}$ with stiffness matrix $K_{ij} = a(\phi_j,\phi_i)$ and load $F_i = \ell(\phi_i)$. **Céa's lemma** then bounds the error:

$$
\|u - u_h\|_V \;\le\; \frac{C}{\alpha}\inf_{v_h \in V_h}\|u - v_h\|_V,
$$

i.e., the Galerkin solution is *quasi-optimal* in $V_h$.

### 3.3 Strengths and Limitations of FEM

* **Strengths.** Geometric flexibility via unstructured meshes, rigorous a-priori and a-posteriori error bounds (Babuška, Brezzi), natural treatment of Neumann conditions through the integration by parts.
* **Limitations.**
  * Mesh generation in 3-D for complex geometries is itself a research field.
  * Stiffness assembly cost scales as $\mathcal{O}(N \log N)$ at best (with multigrid).
  * Adaptive refinement (AMR) requires sophisticated data structures.
  * Strong dependence on quality of triangulation: ill-shaped elements destroy conditioning of $K$.
  * Curse of dimensionality remains: in $d=10$ phase-space PDEs (Boltzmann, Vlasov, Fokker–Planck), FEM is computationally infeasible.

## 4. The Curse of the Mesh

Both FDM and FEM are **mesh-based**: the solution $u$ is represented by a finite vector of values associated with geometric entities (nodes, cells, edges). This has three deep consequences which motivate the PINN paradigm.

1. **Dimensional explosion.** A regular mesh with $N$ nodes per axis in $d$ dimensions requires $N^d$ degrees of freedom. For the high-dimensional PDEs of statistical mechanics, finance, and reinforcement-learning value functions, $d \ge 50$ is routine; even $N=10$ then gives $10^{50}$ DoFs. This is Bellman's *curse of dimensionality* in its most stark form.

2. **Data assimilation is unnatural.** Real-world measurements rarely arrive on a grid. Incorporating sparse, noisy observations into a meshed solver requires Kalman filters, 4D-Var, or ad hoc projections — none native to the discretization.

3. **Inverse problems are bolted on.** Estimating physical parameters from data requires nested loops: solve forward, compute sensitivity, update parameter. Each forward solve is a full mesh computation.

A natural ambition follows: *can we represent $u$ as a continuous, differentiable function on $\Omega \times [0,T]$ directly, with parameters tunable by gradient descent, and let the PDE residual itself drive the fit?*  Answering this requires two ingredients: an expressive parametric family, and a calculus to differentiate composites of that family.  The first is provided by neural networks; the second, by automatic differentiation.

## 5. Neural Networks as Continuous Function Approximators

A feedforward neural network with $L$ hidden layers and activation $\sigma$ defines a parametric map $u_\theta : \mathbb{R}^{d+1} \to \mathbb{R}$ by the recursion

$$
h^{(0)} = (x,t), \qquad h^{(\ell)} = \sigma\!\left(W^{(\ell)} h^{(\ell-1)} + b^{(\ell)}\right) \;\; \ell=1,\dots,L, \qquad u_\theta(x,t) = W^{(L+1)} h^{(L)} + b^{(L+1)},
$$

with parameters $\theta = \{W^{(\ell)}, b^{(\ell)}\}$. The activation $\sigma$ is chosen *smooth* (e.g. $\tanh$, $\text{SiLU}$, $\sin$) to ensure $u_\theta \in C^\infty(\mathbb{R}^{d+1})$ — a property indispensable for evaluating high-order differential operators.

Crucially, $u_\theta$ is defined on the **entire continuum** $\mathbb{R}^{d+1}$ at the cost of $|\theta|$ parameters — orders of magnitude fewer than a high-dimensional mesh. There is no notion of *element*, no Voronoi cell, no FFT grid. The representation is *mesh-free* by construction.

But why should so simple a parametric family suffice? The answer is the celebrated Universal Approximation Theorem.

## 6. The Universal Approximation Theorem (UAT)

**Theorem (Cybenko 1989, Hornik 1991, Pinkus 1999).** Let $\sigma : \mathbb{R} \to \mathbb{R}$ be a non-polynomial continuous function. Then the family

$$
\mathcal{F}_\sigma \;:=\; \left\{\, x \mapsto \sum_{j=1}^N c_j\,\sigma\!\left(w_j^\top x + b_j\right) \,:\, N \in \mathbb{N},\; w_j\in\mathbb{R}^d,\; b_j,c_j\in\mathbb{R}\,\right\}
$$

is dense in $C(K)$ for every compact $K \subset \mathbb{R}^d$ in the sup-norm. That is, for every $f \in C(K)$ and every $\varepsilon > 0$ there exists $\hat f \in \mathcal{F}_\sigma$ with $\sup_{x\in K}|f(x) - \hat f(x)| < \varepsilon$.

### 6.1 Proof Sketch (Cybenko, via Hahn–Banach)

Suppose for contradiction that the closure $\overline{\mathcal{F}_\sigma}$ in $C(K)$ is a proper subspace. By the Hahn–Banach theorem there exists a nonzero continuous linear functional $\Lambda \in C(K)^\ast$ vanishing on $\mathcal{F}_\sigma$. By the Riesz–Markov representation theorem, $\Lambda$ corresponds to a finite signed Borel measure $\mu$ on $K$:

$$
\Lambda(\sigma(w^\top \cdot + b)) \;=\; \int_K \sigma(w^\top x + b)\,d\mu(x) \;=\; 0 \qquad \forall w,b.
$$

If $\sigma$ is *discriminatory* — a property satisfied by any non-polynomial continuous function (Leshno–Lin–Pinkus–Schocken 1993) — this forces $\mu = 0$, contradicting $\Lambda \ne 0$.  $\blacksquare$

### 6.2 Quantitative Versions and Sobolev Approximation

The qualitative UAT does not bound $N$ in terms of $\varepsilon$. Subsequent work fills this gap:

* **Barron (1993).** Functions in the Barron space $\mathcal{B}$ (Fourier transform with finite first moment) are approximated in $L^2$ at rate $\mathcal{O}(N^{-1/2})$ *independent of dimension* — a powerful statement contrasting the $\mathcal{O}(N^{-1/d})$ rates of mesh methods.
* **Hornik–Stinchcombe–White (1989).** Approximation extends to Sobolev spaces $W^{k,p}$, ensuring derivatives also converge — crucial for PINNs where we must approximate $u$, $\nabla u$, $\Delta u$ simultaneously.
* **Yarotsky (2017) / DeVore (2021).** Sharp depth-vs-width trade-offs for ReLU networks approximating $C^k$ functions.

It is the *Sobolev* generalization that licenses PINNs: a smooth neural network can approximate not only $u$, but every derivative $D^\alpha u$ appearing in $\mathcal{N}[u]$, jointly and uniformly on compact subsets.

## 7. Function Spaces and the Geometry of Solutions

PINNs operate not on $\mathbb{R}^N$ but on subsets of *infinite-dimensional function spaces*. To reason rigorously we need the standard hierarchy.

### 7.1 Lebesgue and Sobolev Spaces

* $L^p(\Omega)$: equivalence classes of measurable functions with $\|u\|_p = \big(\int_\Omega |u|^p\big)^{1/p} < \infty$.
* $W^{k,p}(\Omega)$: $u \in L^p$ whose weak derivatives $D^\alpha u$ exist in $L^p$ for $|\alpha| \le k$; equipped with the norm
  $$\|u\|_{W^{k,p}} = \Big(\textstyle\sum_{|\alpha|\le k}\|D^\alpha u\|_p^p\Big)^{1/p}.$$
* $H^k(\Omega) := W^{k,2}(\Omega)$ is a *Hilbert space* with inner product $\langle u,v\rangle_{H^k} = \sum_{|\alpha|\le k} \int_\Omega D^\alpha u\,D^\alpha v$.

### 7.2 Why Hilbert Space?

Hilbert structure brings three indispensable goods:

1. **Orthogonal projection.** Best approximation of $u$ from a closed subspace $V_h$ is the orthogonal projection $P_h u$. This underwrites Galerkin / VPINN methods.
2. **Riesz representation.** Continuous linear functionals on $H^k$ are inner products with elements of $H^k$, giving the bridge between strong and weak forms.
3. **Spectral theory.** Self-adjoint operators on Hilbert spaces (e.g. $-\Delta$ with Dirichlet boundary) admit complete eigen-bases; this anchors the analysis of *spectral bias* in PINN training (Notebook 3).

### 7.3 Operator Embeddings

A PDE $\mathcal{N}: H^k \to H^{k-m}$ (with $m$ the order of $\mathcal{N}$) is a (possibly nonlinear) operator between Hilbert spaces. Approximating $u$ by a neural net $u_\theta$ replaces the search over an infinite-dimensional manifold with optimization on a finite-dimensional parameter manifold $\Theta \subset \mathbb{R}^{|\theta|}$. The continuous embedding

$$
\Phi : \Theta \hookrightarrow H^k(\Omega), \qquad \theta \mapsto u_\theta,
$$

is the geometric content of PINNs: the network parametrizes a smooth, finite-dimensional submanifold of $H^k$, and training is gradient descent on the pullback functional $\theta \mapsto J(u_\theta)$.

## 8. From Strong Form to Optimization — The Calculus of Variations

Many PDEs are Euler–Lagrange equations of an action functional. The Hamiltonian principle and the calculus of variations thus provide a *native* path from physics to optimization.

### 8.1 Action Functionals

Consider a Lagrangian density $\mathcal{L}(u, \nabla u, x)$ and the action

$$
S[u] = \int_\Omega \mathcal{L}(u, \nabla u, x)\,d\Omega.
$$

The first variation $\delta S[u;\eta] = \tfrac{d}{d\epsilon}\big|_0 S[u+\epsilon\eta]$ vanishes for all admissible $\eta$ iff

$$
\boxed{\frac{\partial \mathcal{L}}{\partial u} \;-\; \nabla \cdot \frac{\partial \mathcal{L}}{\partial(\nabla u)} \;=\; 0,}
$$

the Euler–Lagrange equation. Classical mechanics, classical and quantum field theory, and elasticity all flow from this principle.

### 8.2 Example — Dirichlet Energy and Poisson's Equation

For $\mathcal{L} = \tfrac{1}{2}|\nabla u|^2 - f u$, the Euler–Lagrange equation is $-\Delta u = f$. The Dirichlet energy

$$
E[u] = \tfrac{1}{2}\int_\Omega |\nabla u|^2\,d\Omega - \int_\Omega f u\,d\Omega
$$

is convex and coercive on $H^1_0$; its unique minimizer is the solution of Poisson's problem. Minimizing $E[u_\theta]$ over $\theta$ — a *Deep Ritz Method* — produces a finite-dimensional approximation of the PDE solution without ever invoking the strong form.

### 8.3 Beyond Variational Form

Not every PDE admits an action principle (e.g. the dissipative Navier–Stokes equations). A more flexible philosophy is to construct an *empirical risk* directly from the PDE residual:

$$
J[u] \;=\; \frac{1}{2}\int_{\Omega\times(0,T]} \big|\mathcal{N}[u] - f\big|^2\,d\Omega\,dt \;+\; \text{(boundary/initial penalty)}.
$$

Then $u^\ast \in \arg\min_u J[u]$ satisfies $\mathcal{N}[u^\ast] = f$ a.e. (if attainable). This *strong-form residual minimization* is the heart of the PINN.

## 9. The PINN Ansatz — Embedding Differential Operators in $C^\infty$

We now state the central PINN construction.

### 9.1 Definition

Let $u_\theta : \overline{\Omega}\times[0,T] \to \mathbb{R}^n$ be a smooth neural network. Because every operation in $u_\theta$ is the composition of smooth maps, the differential operator $\mathcal{N}$ may be applied *exactly* (up to floating point) at any point $(x,t)$:

$$
r_\theta(x,t) \;:=\; \mathcal{N}[u_\theta](x,t) - f(x,t).
$$

$r_\theta$ is the **PDE residual field**, itself a smooth function on the continuum.

### 9.2 Sampling Replaces Meshing

Choose finite collocation sets:

* $\{(x^r_i,t^r_i)\}_{i=1}^{N_r} \subset \Omega\times(0,T]$ — *residual / interior points*,
* $\{(x^b_i,t^b_i)\}_{i=1}^{N_b} \subset \partial\Omega\times[0,T]$ — *boundary points*,
* $\{x^0_i\}_{i=1}^{N_0} \subset \Omega$ — *initial points*,
* (optional) $\{(x^d_i,t^d_i, u^d_i)\}_{i=1}^{N_d}$ — *measurement / data*.

These need not lie on any lattice and may be drawn from quasi-Monte Carlo sequences (Sobol, Halton, Latin Hypercube). The PINN loss is

$$
\mathcal{L}_{\text{PINN}}(\theta) \;=\; \lambda_r \mathcal{L}_r + \lambda_b \mathcal{L}_b + \lambda_0 \mathcal{L}_0 + \lambda_d \mathcal{L}_d,
$$

where

$$
\mathcal{L}_r(\theta) = \frac{1}{N_r}\sum_{i=1}^{N_r} \big|r_\theta(x^r_i,t^r_i)\big|^2, \quad
\mathcal{L}_b(\theta) = \frac{1}{N_b}\sum_i \big|\mathcal{B}[u_\theta](x^b_i,t^b_i) - g(x^b_i,t^b_i)\big|^2,
$$
$$
\mathcal{L}_0(\theta) = \frac{1}{N_0}\sum_i \big|u_\theta(x^0_i,0) - u_0(x^0_i)\big|^2, \quad
\mathcal{L}_d(\theta) = \frac{1}{N_d}\sum_i \big|u_\theta(x^d_i,t^d_i) - u^d_i\big|^2.
$$

### 9.3 The Three Conceptual Shifts

| Classical | PINN |
|---|---|
| Discrete unknowns at mesh nodes | Continuous parametric function $u_\theta$ |
| Linear/nonlinear algebra solve | Gradient-based minimization in $\Theta$ |
| Derivatives via stencils | Derivatives via automatic differentiation |
| Forward and inverse problems are distinct algorithms | Both are special cases of one optimization |
| Data and physics are reconciled post hoc | Data and physics share a single loss |

### 9.4 What the PINN Is Doing, Mathematically

The PINN search is

$$
\theta^\ast \in \arg\min_{\theta\in\Theta} \mathbb{E}_{(x,t)\sim\mu}\!\big[\,|r_\theta(x,t)|^2\,\big] + \text{penalties},
$$

where $\mu$ is the sampling measure on $\Omega\times(0,T]$. As $N_r \to \infty$, the empirical mean converges (Glivenko–Cantelli / Monte Carlo) to the population mean, i.e. the *strong-form residual squared $L^2$-norm*. Universal approximation in $W^{m,2}$ ensures $\inf_\theta J(u_\theta) \to 0$ as the network capacity grows. The PINN is thus a **Monte Carlo estimator of a Sobolev residual functional**, minimized by stochastic gradient methods on a smooth parametric submanifold of $H^m$.

## 10. A First Look at the Composite Functional

Define the population (continuum) functional

$$
J[u] \;=\; \tfrac{1}{2}\|\mathcal{N}[u] - f\|^2_{L^2(\Omega\times(0,T])} + \tfrac{\beta_b}{2}\|\mathcal{B}u - g\|^2_{L^2(\partial\Omega\times[0,T])} + \tfrac{\beta_0}{2}\|u(\cdot,0) - u_0\|^2_{L^2(\Omega)}.
$$

Provided the PDE is *well-posed* (Hadamard: existence, uniqueness, continuous dependence on data), $J$ has a unique zero in the solution space — namely the classical solution $u^\star$. The PINN training problem is the *parametric pullback*

$$
\min_{\theta} J(u_\theta) \;\approx\; \min_\theta \mathcal{L}_{\text{PINN}}(\theta).
$$

Note that even though $J[u]\ge 0$ with unique zero at $u^\star$, the **pullback** $\theta\mapsto J(u_\theta)$ is in general non-convex with many local minima; this is the source of the optimization pathologies discussed in Notebook 3. The composite structure with multiple terms also raises the question of *loss weighting* — the $\lambda$'s are not innocent. We return to this in depth later.

## 11. Convergence, Consistency, and the Limits of the Paradigm

### 11.1 Convergence in Theory

Mishra & Molinaro (2021), Shin–Darbon–Karniadakis (2020), De Ryck–Mishra (2022) give the following meta-statement: under regularity of $\mathcal{N}$, well-posedness of the PDE, and sufficient capacity of $u_\theta$, the PINN minimizer $u_{\theta^\ast}$ converges in $H^m$ to the true solution as collocation $N_r\to\infty$ and network width/depth $\to\infty$, with explicit error decompositions

$$
\|u_{\theta^\ast} - u^\star\|_{H^m} \;\le\; \underbrace{\varepsilon_{\text{approx}}}_{\text{UAT}} + \underbrace{\varepsilon_{\text{gen}}}_{\text{Monte Carlo}} + \underbrace{\varepsilon_{\text{opt}}}_{\text{training residual}}.
$$

Each term reflects a different mathematical concern: representation power, statistical sampling, and optimization geometry.

### 11.2 Where PINNs Beat — and Lose to — Mesh Methods

* **PINN advantages:** mesh-free in complex/high-dim geometries, native data assimilation, unified forward/inverse, automatic regularization via the network prior.
* **PINN disadvantages:** slow training compared to one-shot linear solves on well-posed elliptic problems, sensitivity to loss weights, spectral bias against high-frequency modes, no rigorous a-posteriori error bounds (yet) competitive with FEM's.

PINNs do **not** universally replace FEM/FDM. They occupy a complementary niche: PDE-constrained data assimilation, parameterized PDE families (operator learning), high-dimensional problems, and rapid prototyping in messy geometries.

### 11.3 The Roadmap Ahead

Having established that smooth neural networks form a continuous submanifold of $H^m$ on which a Sobolev residual functional can be minimized, three concrete questions remain.

1. **How exactly are the derivatives $\partial_t u_\theta, \Delta u_\theta, \dots$ computed at machine precision and finite cost?** — Notebook 2 (Automatic Differentiation).
2. **Why does the optimization land in good minima, and what pathologies obstruct it?** — Notebook 3 (Loss landscape, NTK, spectral bias, two-stage optimization).
3. **How do we extend the basic ansatz to enforce conservation, exploit weak forms, scale to large domains, and quantify uncertainty?** — Notebook 4 (cPINNs, VPINNs, XPINNs, B-PINNs).

We will then unify everything in Notebook 5 by carefully formulating the 1-D viscous Burgers' equation as the canonical PINN benchmark, ready for code.

## References for Further Study

1. **Raissi, M., Perdikaris, P., Karniadakis, G. E.** *Physics-informed neural networks: A deep learning framework for solving forward and inverse problems involving nonlinear partial differential equations.* J. Comp. Phys. 378 (2019), 686–707.
2. **Cybenko, G.** *Approximation by superpositions of a sigmoidal function.* Math. Control Signals Systems 2 (1989), 303–314.
3. **Hornik, K., Stinchcombe, M., White, H.** *Multilayer feedforward networks are universal approximators.* Neural Networks 2 (1989), 359–366.
4. **Pinkus, A.** *Approximation theory of the MLP model in neural networks.* Acta Numerica 8 (1999).
5. **Barron, A. R.** *Universal approximation bounds for superpositions of a sigmoidal function.* IEEE Trans. Inf. Theory 39 (1993).
6. **Brenner, S., Scott, L. R.** *The Mathematical Theory of Finite Element Methods.* Springer, 2008.
7. **Evans, L. C.** *Partial Differential Equations.* AMS Graduate Studies in Mathematics, 2010.
8. **Mishra, S., Molinaro, R.** *Estimates on the generalization error of physics-informed neural networks.* IMA J. Numer. Anal. 42 (2022).
9. **E, W., Yu, B.** *The Deep Ritz method: A deep learning-based numerical algorithm for solving variational problems.* Commun. Math. Stat. 6 (2018), 1–12.